# RAMS Semantic-Judge Evaluation

This notebook measures whether `deepseek-v4-pro` can judge a proposed RAMS event-argument pair from document context. The test split is document-isolated and contains 998 records: 499 valid candidates paired with 499 controlled negatives.

Each request includes the document, target event, permitted roles, and candidate `(role, span)` only. Gold labels and corruption types are kept outside the prompt to prevent evaluation leakage. The workflow is ordered as configuration, data inspection, prompt preview, reproducible run setup, resumable execution, and error analysis.


In [ ]:
# 1. Configure the semantic-judge evaluation.
SAMPLE_LIMIT = None  # None evaluates all 998 test records; use a small integer only for a smoke run.
RUN_EVALUATION = True  # Set to False to inspect data, prompts, and paths without making API calls.
RUN_NAME = None  # The default deterministic name prevents accidental mixing of incompatible runs.

# Model and prompt settings reproduce the final judge condition.
MODEL_NAME = "deepseek-v4-pro"
PROMPT_VERSION = "v5_fs10"  # Compact prompt with ten fixed demonstrations.
DEEPSEEK_THINKING = "enabled"  # Reasoning is enabled for the final semantic decision run.
REASONING_EFFORT = "high"  # High was retained as the accuracy-cost compromise; max is optional.
MAX_WORKERS = 8  # Provides useful throughput while keeping concurrent API pressure bounded.
MAX_TOKENS = 4096  # Covers reasoning plus the short binary judgement without truncation.
TIMEOUT_SECONDS = 120.0  # Allows difficult documents to finish before a request is retried.
RETRY_TIMES = 3  # Handles transient service failures while keeping retry behavior finite.
RETRY_FAILED = True  # Resume runs by retrying only missing, invalid, or failed predictions.

if DEEPSEEK_THINKING not in {"enabled", "disabled"}:
    raise ValueError("DEEPSEEK_THINKING must be 'enabled' or 'disabled'")
if DEEPSEEK_THINKING == "enabled" and REASONING_EFFORT not in {"high", "max"}:
    raise ValueError("REASONING_EFFORT must be 'high' or 'max' when thinking is enabled")
if DEEPSEEK_THINKING == "disabled" and REASONING_EFFORT is not None:
    raise ValueError("REASONING_EFFORT must be None when thinking is disabled")

# Paths remain project-relative so the notebook works from the repository root or notebooks directory.
API_KEY_FILE = "deepseek_api.txt"  # Local secret; never saved with the run artifacts.
DATASET_FILE = "Data/RAMS_judge/splits/doc_level_v1/test.jsonl"  # Final document-level test split.
PROMPT_FILE = f"prompts/rams_judge_{PROMPT_VERSION}.txt"
OUTPUT_ROOT = "results/rams_judge_evaluation"


In [ ]:
# 2. Resolve project paths, load the requested records, and summarize class balance.
# The selection summary should be checked before execution, especially when SAMPLE_LIMIT is not None.
import hashlib
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.rams_judge_evaluator import (
    build_judge_prompt,
    compute_metrics,
    load_jsonlines,
    load_latest_results,
    load_prompt_template,
    run_evaluation,
    save_comparison_outputs,
    save_json,
    select_records,
    summarize_selection,
)

dataset_path = PROJECT_ROOT / DATASET_FILE
prompt_path = PROJECT_ROOT / PROMPT_FILE
api_key_path = PROJECT_ROOT / API_KEY_FILE
output_root = PROJECT_ROOT / OUTPUT_ROOT

all_records = load_jsonlines(dataset_path)
selected_records = select_records(all_records, SAMPLE_LIMIT)
selection_summary = summarize_selection(selected_records)
prompt_template = load_prompt_template(prompt_path)

print(json.dumps(selection_summary, ensure_ascii=False, indent=2))
print(f"Dataset: {dataset_path}")
print(f"Prompt: {prompt_path}")


In [ ]:
# 3. Preview the exact prompt for one record without calling the model.
# Gold metadata is printed separately to verify that neither the label nor corruption type enters the prompt.
preview_record = selected_records[0]
preview_prompt = build_judge_prompt(preview_record, prompt_template)
print(preview_prompt)
print("\nHidden gold metadata (not sent to DeepSeek):")
print({
    "id": preview_record["id"],
    "label": preview_record["label"],
    "case_type": preview_record["case_type"],
})


In [ ]:
# 4. Build a stable run directory and freeze all settings required for reproducibility.
# Dataset and prompt hashes prevent an existing RUN_NAME from silently reusing changed inputs.
limit_label = "all" if SAMPLE_LIMIT is None else f"n{len(selected_records)}"
thinking_label = (
    f"thinking-{REASONING_EFFORT}"
    if DEEPSEEK_THINKING == "enabled"
    else "thinking-disabled"
)
dataset_label = dataset_path.stem
active_run_id = RUN_NAME or (
    f"rams_judge_{dataset_label}_{PROMPT_VERSION}_{MODEL_NAME}_{thinking_label}_{limit_label}"
)
run_dir = output_root / active_run_id
results_path = run_dir / "predictions.jsonl"
metrics_path = run_dir / "metrics.json"
config_path = run_dir / "config.json"
prompt_snapshot_path = run_dir / "prompt_snapshot.txt"

run_config = {
    "run_id": active_run_id,
    "model": MODEL_NAME,
    "prompt_version": PROMPT_VERSION,
    "sample_limit": SAMPLE_LIMIT,
    "selected_count": len(selected_records),
    "selection_summary": selection_summary,
    "dataset_path": str(dataset_path),
    "dataset_sha256": hashlib.sha256(dataset_path.read_bytes()).hexdigest(),
    "prompt_path": str(prompt_path),
    "prompt_sha256": hashlib.sha256(prompt_path.read_bytes()).hexdigest(),
    "prompt_snapshot_path": str(prompt_snapshot_path),
    "max_workers": MAX_WORKERS,
    "max_tokens": MAX_TOKENS,
    "timeout_seconds": TIMEOUT_SECONDS,
    "retry_times": RETRY_TIMES,
    "retry_failed": RETRY_FAILED,
    "thinking": DEEPSEEK_THINKING,
    "reasoning_effort": REASONING_EFFORT,
    "temperature": 0 if DEEPSEEK_THINKING == "disabled" else None,
}
if config_path.exists():
    existing_config = json.loads(config_path.read_text(encoding="utf-8"))
    invariant_keys = (
        "model",
        "prompt_version",
        "sample_limit",
        "selected_count",
        "dataset_sha256",
        "prompt_sha256",
        "thinking",
        "reasoning_effort",
    )
    mismatches = {
        key: {"existing": existing_config.get(key), "current": run_config.get(key)}
        for key in invariant_keys
        if existing_config.get(key) != run_config.get(key)
    }
    if mismatches:
        raise ValueError(f"RUN_NAME configuration mismatch: {mismatches}")
else:
    save_json(config_path, run_config)

if prompt_snapshot_path.exists():
    if prompt_snapshot_path.read_text(encoding="utf-8") != prompt_template:
        raise ValueError("RUN_NAME prompt snapshot differs from the current prompt")
else:
    prompt_snapshot_path.parent.mkdir(parents=True, exist_ok=True)
    prompt_snapshot_path.write_text(prompt_template, encoding="utf-8")

print(f"Run ID: {active_run_id}")
print(f"Results: {results_path}")
print(f"Config: {config_path}")
print(f"Thinking: {DEEPSEEK_THINKING}; reasoning effort: {REASONING_EFFORT}")


In [ ]:
# 5. Run or resume the batch evaluation; this is the only cell that makes model-service calls.
# Existing valid predictions are reused, and each new result is persisted incrementally for recovery.
if not RUN_EVALUATION:
    print("RUN_EVALUATION=False: no model calls were made. Review the configuration and prompt before enabling the run.")
else:
    from tqdm.auto import tqdm
    from src.kg_evaluator import DeepSeekJudgeClient
    from src.rams_judge_evaluator import save_comparison_outputs

    if not api_key_path.exists():
        raise FileNotFoundError(f"DeepSeek API key file not found: {api_key_path}")

    judge_client = DeepSeekJudgeClient(
        api_key_path=api_key_path,
        model=MODEL_NAME,
        max_tokens=MAX_TOKENS,
        thinking=DEEPSEEK_THINKING,
        reasoning_effort=REASONING_EFFORT,
        timeout=TIMEOUT_SECONDS,
        retry_times=RETRY_TIMES,
    )

    progress = tqdm(total=len(selected_records), desc="DeepSeek RAMS judge")
    existing = load_latest_results(results_path)
    selected_ids = {record["id"] for record in selected_records}
    reusable_count = sum(
        record_id in selected_ids
        and (not RETRY_FAILED or isinstance(result.get("predicted_label"), bool))
        for record_id, result in existing.items()
    )
    progress.update(reusable_count)

    try:
        run_results = run_evaluation(
            selected_records,
            client=judge_client,
            prompt_template=prompt_template,
            results_path=results_path,
            max_workers=MAX_WORKERS,
            retry_failed=RETRY_FAILED,
            progress_callback=lambda _result: progress.update(1),
        )
    finally:
        progress.close()

    if len(run_results) != len(selected_records):
        raise RuntimeError(
            f"Result count mismatch: {len(run_results)} / {len(selected_records)}"
        )

    run_metrics = compute_metrics(run_results)
    save_json(metrics_path, run_metrics)
    comparison_paths = save_comparison_outputs(
        selected_records,
        run_results,
        run_dir,
    )
    print(json.dumps(run_metrics, ensure_ascii=False, indent=2))
    print(f"Metrics saved to: {metrics_path}")
    print({name: str(path) for name, path in comparison_paths.items()})


In [ ]:
# 6. Inspect saved progress and recompute final outputs once every selected record is available.
# Using the saved JSONL makes this cell safe to rerun after interruption without repeating inference.
from src.rams_judge_evaluator import save_comparison_outputs

latest_results = load_latest_results(results_path)
available_results = [
    latest_results[record["id"]]
    for record in selected_records
    if record["id"] in latest_results
]
print(f"Available results: {len(available_results)} / {len(selected_records)}")

if len(available_results) == len(selected_records):
    final_metrics = compute_metrics(available_results)
    save_json(metrics_path, final_metrics)
    comparison_paths = save_comparison_outputs(
        selected_records,
        available_results,
        run_dir,
    )
    print(json.dumps(final_metrics, ensure_ascii=False, indent=2))
    print({name: str(path) for name, path in comparison_paths.items()})
else:
    print("The run is incomplete; rerun with the same configuration to resume automatically.")


In [ ]:
# 7. Separate service/parsing failures from semantic mistakes and inspect representative cases.
# The first five mistakes retain event, role, span, gold label, prediction, and raw response for diagnosis.
if not available_results:
    print("No prediction results are available.")
else:
    records_by_id = {record["id"]: record for record in selected_records}
    failures = [result for result in available_results if result["error"] is not None]
    mistakes = [result for result in available_results if not result["correct"]]
    print(f"API/parse failures: {len(failures)}")
    print(f"Strict mistakes: {len(mistakes)}")

    for result in mistakes[:5]:
        source = records_by_id[result["id"]]
        print("\n---")
        print({
            "id": result["id"],
            "case_type": result["case_type"],
            "event_type": source["event"]["type"],
            "trigger": source["event"]["trigger"]["text"],
            "candidate_role": source["candidate"]["role"],
            "candidate_span": source["candidate"]["span"]["text"],
            "gold": result["gold_label"],
            "predicted": result["predicted_label"],
            "error": result["error"],
            "raw_output": result["raw_output"],
        })


## Scoring protocol

Primary metrics are accuracy, positive-class precision/recall/F1, and macro-F1. Per-type accuracy reports the acceptance rate for valid candidates and the rejection rates for `role_corruption` and `span_swap` negatives.

API and parsing failures remain in strict scoring: a positive candidate becomes a false negative and a negative candidate becomes a false positive. `coverage` and `valid_response_accuracy` are reported separately as stability diagnostics.

Each run stores `predictions.jsonl`, aggregate `metrics.json`, a row-level `comparison.csv`, a compact `mistakes.jsonl`, the validated configuration, and the exact prompt snapshot. These files distinguish model quality from incomplete coverage and make later comparisons reproducible.
